In [7]:
# utils/processing.py
from typing import Dict
import pandas as pd
from utils.exception_handler import handle_exception_wrap
import datetime
import logging

logger = logging.getLogger(__name__)

logging.basicConfig(level=logging.DEBUG)


def is_readme_sheet(sheet_name: str) -> bool:
    normalized = sheet_name.lower().replace(' ', '')
    return 'readme' in normalized


def create_report(url: str, resource_id: str = None, download_url: str = None, sample_size: int = 5):
    sampler = DataSampler()
    sheets = sampler.load_from_url(url)
    new_sample_dict = {}
    for name, df in sheets.items():
        logger.debug(f'Processing sheet: {name}')
        if not is_readme_sheet(name):
            new_sample_dict[name] = sampler.sample_dataframe(df, sample_size)
    reports = []
    for sheet_name, column_dict_with_sample_values in new_sample_dict.items():
        sdd_report = {
            'resource_id': resource_id,
            'file_name': url,
            'file_url': download_url,
            'sheet_name': sheet_name,
            'processing_timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'processing_success': True,
            'n_records': len(sheets[sheet_name]),
            'n_columns': len(sheets[sheet_name].columns),
            'completion_tokens': 0,
            'prompt_tokens': 0,
            'pii_sensitive': False,
            'non_pii_sensitive': False,
            'columns': [],
        }
        if 'readme' in sheet_name.lower().replace(' ', ''):
            reports.append(sdd_report)
            continue

        # Add to columns the pii_entity TODO and sensitive False
        for col, item in column_dict_with_sample_values.items():
            sdd_report['columns'].append(
                {
                    'column_name': col,
                    'sample_values': item,
                    'pii': {'entity_type': 'TODO', 'sensitive': False},
                }
            )
        reports.append(sdd_report)
    print(f'Reports: {reports}')
    return reports


class DataSampler:
    """
    Utility class to load a dataset (CSV/XLS/XLSX) directly from a URL and sample random records.
    """

    SUPPORTED_EXTENSIONS = ('.csv', '.xls', '.xlsx')

    @handle_exception_wrap()
    def _validate_url(self, url: str) -> str:
        """Check if the URL points to a supported file type."""
        url_lower = url.lower()
        if not any(url_lower.endswith(ext) for ext in self.SUPPORTED_EXTENSIONS):
            raise ValueError(f'Unsupported file type. Only {", ".join(self.SUPPORTED_EXTENSIONS)} are supported.')
        return url_lower

    @handle_exception_wrap()
    def load_from_url(self, url: str) -> Dict[str, pd.DataFrame]:
        """Load CSV/XLS/XLSX from a URL into a dictionary of DataFrames keyed by sheet name."""
        url = self._validate_url(url)
        if url.endswith('.csv'):
            df = pd.read_csv(url, header=None, nrows=200)

            df = self._concatenate_header(df)
            # Put the most complete rows to the top
            df_sorted = (
                df.assign(num_nans=df.isna().sum(axis=1))
                .sort_values('num_nans', ascending=True)
                .drop(columns='num_nans')
            )
            return {'sheet1': df_sorted}

        # Excel files: can contain multiple sheets
        df_dict = pd.read_excel(url, sheet_name=None, nrows=1000, header=None)
        return {sheet_name: self._concatenate_header(df) for sheet_name, df in df_dict.items()}

    @handle_exception_wrap()
    def _concatenate_header(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Attempts to detect header rows and flatten them into single column names.
        Treats row 0 as the header row and concatenates all header rows including it.
        """
        # If dict, get first sheet
        if isinstance(df, dict):
            df = df[list(df.keys())[0]]

        # Skip leading all-NaN rows
        while not df.empty and df.iloc[0].isna().all():
            df = df.iloc[1:].reset_index(drop=True)

        if df.empty:
            return df

        # Find the end of the header block (first complete row starting from index 0)
        # Header row itself is at index 0, so we look for the last header row
        header_end_row = 0
        for idx in range(len(df)):
            row = df.iloc[idx]
            if row.notna().all():
                header_end_row = idx
                break

        # Extract header block (rows 0 to header_end_row, inclusive)
        # This includes the header row at index 0
        header_block = df.iloc[: header_end_row + 1].fillna('').astype(str)

        # Forward fill missing values within each row, then across rows
        header_block = header_block.apply(lambda row: row.replace('', None).ffill(), axis=1)
        header_block = header_block.replace('', None).ffill()

        # Combine multi-row headers into single column names
        final_columns = header_block.apply(lambda col: ' | '.join([v for v in col if v]), axis=0)
        # Data starts after the header block (header_end_row + 1)
        cleaned_df = df.iloc[header_end_row + 1 :].copy()
        cleaned_df.columns = final_columns
        return cleaned_df.reset_index(drop=True)

    @handle_exception_wrap()
    def sample_dataframe(self, df: pd.DataFrame, sample_size: int = 10) -> dict:
        """Return a dict of column -> sample values, using rows with the most non-null values first."""

        if df.empty:
            return {}

        # Work on a copy to avoid mutating original DF
        df_sorted = df.copy()

        # Add completeness score
        df_sorted['__null_count__'] = df_sorted.isna().sum(axis=1)

        # Sort so best rows (fewest nulls) appear at the top
        df_sorted = df_sorted.sort_values('__null_count__')

        sample_dict = {}

        for col in df.columns:  # only iterate real columns
            col_data = df_sorted[col]

            # Drop empty values (NaN or empty string)
            non_empty = col_data.dropna()
            non_empty = non_empty[non_empty != '']

            # Take the top N most complete values
            print(type(non_empty))
            values = non_empty.head(sample_size).values.ravel().tolist()

            # If the column has no usable values
            if not values:
                values = [''] * sample_size  # or: df[col].unique()[:sample_size]

            # Pad to sample_size
            while len(values) < sample_size:
                values.append('')

            sample_dict[col] = values

        return sample_dict

    @handle_exception_wrap()
    def sample(self, url: str, sample_size: int = 5) -> Dict[str, pd.DataFrame]:
        """Main entrypoint: load and sample dataset(s) from a URL."""
        sheets = self.load_from_url(url)

        new_sample_dict = {}
        for name, df in sheets.items():
            if not is_readme_sheet(name):
                new_sample_dict[name] = self.sample_dataframe(df, sample_size)
        return new_sample_dict

In [8]:
sdd_report = create_report('research/data/bgd_dataset_joint-msna_refugee_september-2019.xlsx')

DEBUG:__main__:Processing sheet: READ ME
DEBUG:__main__:Processing sheet: Dataset - Household
DEBUG:__main__:Processing sheet: Dataset - Individual
DEBUG:__main__:Processing sheet: Survey Weights
DEBUG:__main__:Processing sheet: Data Key - HH
DEBUG:__main__:Processing sheet: Data Key - Individual
DEBUG:__main__:Processing sheet: Kobo tool - survey
DEBUG:__main__:Processing sheet: Kobo tool - choices
DEBUG:__main__:Processing sheet: Cleaning Log


<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.S

In [9]:
sdd_report

[{'resource_id': None,
  'file_name': 'research/data/bgd_dataset_joint-msna_refugee_september-2019.xlsx',
  'file_url': None,
  'sheet_name': 'Dataset - Household',
  'processing_timestamp': '2025-12-20 16:37:24',
  'processing_success': True,
  'n_records': 999,
  'n_columns': 536,
  'completion_tokens': 0,
  'prompt_tokens': 0,
  'pii_sensitive': False,
  'non_pii_sensitive': False,
  'columns': [{'column_name': 'UUID',
    'sample_values': ['12998207-f4c2-47e5-b8ff-9dee96e285f6',
     '4f87a177-5124-473e-af85-f38177a16c8a',
     '492dcb7f-b5d3-48ef-899d-dd5e441f829e',
     '844d8eb7-7171-4fd5-a39c-3834ecf28ed3',
     '8b1b6597-4dc6-4938-89c3-4ac7d5edcb1d'],
    'pii': {'entity_type': 'TODO', 'sensitive': False}},
   {'column_name': 'survey_date',
    'sample_values': [datetime.datetime(2019, 8, 19, 0, 0),
     datetime.datetime(2019, 8, 17, 0, 0),
     datetime.datetime(2019, 8, 18, 0, 0),
     datetime.datetime(2019, 8, 6, 0, 0),
     datetime.datetime(2019, 8, 19, 0, 0)],
    'pii

In [ ]:
# For each print key
for sheet in sdd_report:
    print(sheet['sheet_name'])

Dataset - Household
Dataset - Individual
Survey Weights
Data Key - HH
Data Key - Individual
Kobo tool - survey
Kobo tool - choices
Cleaning Log


In [ ]:
# Load excel
import pandas as pd

df = pd.read_excel(
    'research/data/bgd_dataset_joint-msna_refugee_september-2019.xlsx', header=None, sheet_name=None, nrows=1000
)
print(df.keys())
print(df['READ ME'].head())

readme_df = df['READ ME']
print(type(readme_df))
# Print shape of df
print(readme_df.shape)

dict_keys(['READ ME', 'Dataset - Household', 'Dataset - Individual', 'Survey Weights', 'Data Key - HH', 'Data Key - Individual', 'Kobo tool - survey', 'Kobo tool - choices', 'Cleaning Log'])
                                                   0  \
0  2019 Joint Multi-Sector Needs Assessment (MSNA...   
1                                              Items   
2                                 Project Background   
3                     Primary data collection period   
4                                       Methodology    

                                                   1   2  \
0                                                NaN NaN   
1                                        Description NaN   
2  In successive waves over four decades, Rohingy... NaN   
3  Household data collection took place from 5 Au... NaN   
4  A total of 3,418 households, composed of 17,16... NaN   

                                                   3  
0                                                NaN  
1

In [42]:
first_df = df['Data Key - HH']
first_df

KeyError: 'Data Key - HH'

In [ ]:
def drop_leading_all_none_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove sequential top rows where all values are None/NaN.
    """
    # Boolean mask: True for rows that are all None/NaN
    all_none_mask = df.isna().all(axis=1)

    # Find index where first non-all-None row appears
    first_valid_idx = all_none_mask.idxmin() if all_none_mask.any() else df.index[0]

    # If the very first row is not all None, return original df
    if not all_none_mask.iloc[0]:
        return df

    # Drop rows up to (but not including) first valid row
    return df.loc[first_valid_idx:].reset_index(drop=True)


trimmed_df = drop_leading_all_none_rows(first_df)
trimmed_df

,0,1,2,3,4,5,6
0,NaN,Column Keys,NaN,NaN,Option Keys,NaN,NaN
1,NaN,Column,Question,NaN,Column,Option,Value
2,NaN,survey_date,Date survey was conducted,NaN,camp_name,camp1e,Camp 1E
3,NaN,NaN,NaN,NaN,NaN,camp 1w,Camp 1W
4,NaN,NaN,NaN,NaN,NaN,camp2e,Camp 2E
...,...,...,...,...,...,...,...
456,NaN,NaN,NaN,NaN,NaN,bamboo,Bamboo
457,NaN,NaN,NaN,NaN,NaN,tarpaulin,Tarpaulin
458,NaN,NaN,NaN,NaN,NaN,wood,Wood
459,NaN,NaN,NaN,NaN,NaN,tin,Tin


,1,2,3,4,5,6
0,Column Keys,NaN,NaN,Option Keys,NaN,NaN
1,Column,Question,NaN,Column,Option,Value
2,survey_date,Date survey was conducted,NaN,camp_name,camp1e,Camp 1E
3,NaN,NaN,NaN,NaN,camp 1w,Camp 1W
4,NaN,NaN,NaN,NaN,camp2e,Camp 2E
...,...,...,...,...,...,...
456,NaN,NaN,NaN,NaN,bamboo,Bamboo
457,NaN,NaN,NaN,NaN,tarpaulin,Tarpaulin
458,NaN,NaN,NaN,NaN,wood,Wood
459,NaN,NaN,NaN,NaN,tin,Tin


In [ ]:
def multiple_tables_present(df: pd.DataFrame) -> bool:
    """
    Check if the dataframe contains multiple tables by counting how many columns there are with only none values
    """
    count = df.isna().all(axis=0).sum()
    return count > 1


print(f'Are there multiple tables present? {multiple_tables_present(trimmed_df)}')


def drop_leading_all_none_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove sequential leftmost columns where all values are None/NaN.
    """
    # Boolean mask: True for columns that are all None/NaN
    all_none_mask = df.isna().all(axis=0)

    # If first column is not all None, return original df
    if not all_none_mask.iloc[0]:
        return df

    # Find index of first column that is NOT all None
    first_valid_col = all_none_mask.idxmin()

    # Slice from first valid column onward
    return df.loc[:, first_valid_col:].reset_index(drop=True)


trimmed_df = drop_leading_all_none_columns(trimmed_df)
trimmed_df


def count_tables_present(df: pd.DataFrame) -> int:
    # Get indices of columns with only none values
    none_columns = df.columns[df.isna().all(axis=0)]
    print(none_columns)
    #
    return len(none_columns) + 1


count_tables_present(trimmed_df)

Are there multiple tables present? False
Index([3], dtype='int64')


2

In [ ]:
# Drop all columns that contain only none values
trimmed_df = trimmed_df.dropna(axis=1, how='all')
trimmed_df

,1,2,4,5,6
0,Column Keys,NaN,Option Keys,NaN,NaN
1,Column,Question,Column,Option,Value
2,survey_date,Date survey was conducted,camp_name,camp1e,Camp 1E
3,NaN,NaN,NaN,camp 1w,Camp 1W
4,NaN,NaN,NaN,camp2e,Camp 2E
...,...,...,...,...,...
456,NaN,NaN,NaN,bamboo,Bamboo
457,NaN,NaN,NaN,tarpaulin,Tarpaulin
458,NaN,NaN,NaN,wood,Wood
459,NaN,NaN,NaN,tin,Tin


In [ ]:
from typing import Optional


def first_fully_non_empty_row_index(df: pd.DataFrame) -> Optional[int]:
    """
    Geeft de index terug van de eerste rij waarin GEEN None/NaN voorkomt.
    Als zo'n rij niet bestaat, wordt None teruggegeven.
    """
    fully_filled_mask = ~df.isna().any(axis=1)

    if not fully_filled_mask.any():
        return None

    return fully_filled_mask.idxmax()


header_index = first_fully_non_empty_row_index(trimmed_df)
print(header_index)

1


In [73]:
trimmed_df = trimmed_df.ffill(axis=0)
trimmed_df = trimmed_df.ffill(axis=1)
trimmed_df

,1,2,4,5,6
0,Column Keys,Column Keys,Option Keys,Option Keys,Option Keys
1,Column,Question,Column,Option,Value
2,survey_date,Date survey was conducted,camp_name,camp1e,Camp 1E
3,survey_date,Date survey was conducted,camp_name,camp 1w,Camp 1W
4,survey_date,Date survey was conducted,camp_name,camp2e,Camp 2E
...,...,...,...,...,...
456,wall,What building material was used to construct t...,wall,bamboo,Bamboo
457,wall,What building material was used to construct t...,wall,tarpaulin,Tarpaulin
458,wall,What building material was used to construct t...,wall,wood,Wood
459,wall,What building material was used to construct t...,wall,tin,Tin


In [ ]:
import pandas as pd


def merge_two_string_rows(df: pd.DataFrame, sep: str = ' ') -> pd.DataFrame:
    """
    Verwacht een dataframe met exact twee rijen.
    Voegt per kolom de strings samen tot één rij.
    """

    merged = df.astype(str).agg(sep.join, axis=0)

    return pd.DataFrame([merged.values], columns=df.columns)


# Get the records until index_of_first_fully_non_empty_row
header_df = trimmed_df.iloc[: header_index + 1]

# Merge the header rows into one row
header_row = merge_two_string_rows(header_df)
header_row

,1,2,4,5,6
0,Column Keys Column,Column Keys Question,Option Keys Column,Option Keys Option,Option Keys Value


In [ ]:
# Make the header row the new header of the dataframe
trimmed_df.columns = header_row.values[0]
trimmed_df = trimmed_df.iloc[header_index + 1 :]
trimmed_df

,Column Keys Column,Column Keys Question,Option Keys Column,Option Keys Option,Option Keys Value
3,survey_date,Date survey was conducted,camp_name,camp 1w,Camp 1W
4,survey_date,Date survey was conducted,camp_name,camp2e,Camp 2E
5,camp_name,Name of camp,camp_name,camp2w,Camp 2W
6,informed_consent,To ensure coordination of the needed assistanc...,camp_name,camp3,Camp 3
7,respondent_age,Age of respondent,camp_name,camp4,Camp 4
...,...,...,...,...,...
456,wall,What building material was used to construct t...,wall,bamboo,Bamboo
457,wall,What building material was used to construct t...,wall,tarpaulin,Tarpaulin
458,wall,What building material was used to construct t...,wall,wood,Wood
459,wall,What building material was used to construct t...,wall,tin,Tin
